# CineSense-AI: Recommendation System Evaluation

## Objective

This notebook evaluates the CineSense-AI recommendation system using standard recommender-system metrics.

### Evaluation Metrics
- Precision@K
- Recall@K
- Hit Rate@K
- NDCG@K
- Catalog Coverage
- Recommendation Diversity

The goal is to measure how accurately and effectively CineSense-AI recommends relevant movies to users.

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving movies_enriched.csv to movies_enriched.csv


In [ ]:
import pandas as pd
import numpy as np
import math

from collections import defaultdict

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
ratings = pd.read_csv("ratings_clean.csv")
movies = pd.read_csv("movies_enriched.csv")
movie_genres = pd.read_csv("movie_genres.csv")

print("Ratings:", ratings.shape)
print("Movies:", movies.shape)
print("Movie Genres:", movie_genres.shape)

display(ratings.head())
display(movies.head())
display(movie_genres.head())

Ratings: (100836, 5)
Movies: (9742, 25)
Movie Genres: (22084, 2)


,userId,movieId,rating,timestamp,datetime
0,1,1,4.0,964982703,2000-07-30 18:45:03
1,1,3,4.0,964981247,2000-07-30 18:20:47
2,1,6,4.0,964982224,2000-07-30 18:37:04
3,1,47,5.0,964983815,2000-07-30 19:03:35
4,1,50,5.0,964982931,2000-07-30 18:48:51


,movieId,title,genres,year,clean_title,genre_list,combined_tags,rating_count,average_rating,tmdbId,...,tmdb_popularity,vote_average,vote_count,original_language,poster_path,backdrop_path,adult,status,tagline,tmdb_genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0,Toy Story,"['Adventure', 'Animation', 'Children', 'Comedy...",pixar pixar fun,215,3.92,862.0,...,30.9498,7.983,20174.0,en,/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,/3Rfvhy1Nl6sSGJwyjb0QiZzZYlB.jpg,False,Released,The adventure takes off when toys come to life!,Family|Comedy|Animation|Adventure
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0,Jumanji,"['Adventure', 'Children', 'Fantasy']",fantasy magic board game robin williams game,110,3.43,8844.0,...,2.4359,7.249,11395.0,en,/iWV47r6kFneCiApgrMII5HSkfHw.jpg,/qSxeCfWUUyht9hZgaaYmtPtTkw2.jpg,False,Released,It's a jungle in here.,Adventure|Fantasy|Family
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0,Grumpier Old Men,"['Comedy', 'Romance']",moldy old,52,3.26,15602.0,...,1.6558,6.479,432.0,en,/1FSXpj5e8l4KH6nVFO5SPUeraOt.jpg,/1o4vuCHpmd4DXofMYDUwpnhKiuy.jpg,False,Released,Still Yelling. Still Fighting. Still Ready for...,Romance|Comedy
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0,Waiting to Exhale,"['Comedy', 'Drama', 'Romance']",NaN,7,2.36,31357.0,...,1.8246,6.261,207.0,en,/4wjGMwPsdlvi025ZqR4rXnFDvBz.jpg,/jZjoEKXMTDoZAGdkjhAdJaKtXSN.jpg,False,Released,Friends are the people who let you be yourself...,Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy,1995.0,Father of the Bride Part II,['Comedy'],pregnancy remake,49,3.07,11862.0,...,2.1537,6.271,819.0,en,/rj4LBtwQ0uGrpBnCELr716Qo3mw.jpg,/lEsjVrGU21BeJjF5AF9EWsihDpw.jpg,False,Released,Just when his world is back to normal... he's ...,Comedy|Family


,movieId,genre
0,1,Adventure
1,1,Animation
2,1,Children
3,1,Comedy
4,1,Fantasy


In [ ]:
print("Ratings columns:")
print(ratings.columns.tolist())

print("\nMovies columns:")
print(movies.columns.tolist())

print("\nMovie Genres columns:")
print(movie_genres.columns.tolist())

Ratings columns:
['userId', 'movieId', 'rating', 'timestamp', 'datetime']

Movies columns:
['movieId', 'title', 'genres', 'year', 'clean_title', 'genre_list', 'combined_tags', 'rating_count', 'average_rating', 'tmdbId', 'tmdb_title', 'original_title', 'overview', 'runtime', 'release_date', 'tmdb_popularity', 'vote_average', 'vote_count', 'original_language', 'poster_path', 'backdrop_path', 'adult', 'status', 'tagline', 'tmdb_genres']

Movie Genres columns:
['movieId', 'genre']


In [ ]:
# STEP 2.1 - Identify relevant ratings

RELEVANCE_THRESHOLD = 4.0

relevant_ratings = ratings[
    ratings["rating"] >= RELEVANCE_THRESHOLD
].copy()

print("Total ratings:", len(ratings))
print("Relevant ratings (>= 4.0):", len(relevant_ratings))

print(
    "Percentage relevant:",
    round(len(relevant_ratings) / len(ratings) * 100, 2),
    "%"
)

display(relevant_ratings.head(10))

Total ratings: 100836
Relevant ratings (>= 4.0): 48580
Percentage relevant: 48.18 %


,userId,movieId,rating,timestamp,datetime
0,1,1,4.0,964982703,2000-07-30 18:45:03
1,1,3,4.0,964981247,2000-07-30 18:20:47
2,1,6,4.0,964982224,2000-07-30 18:37:04
3,1,47,5.0,964983815,2000-07-30 19:03:35
4,1,50,5.0,964982931,2000-07-30 18:48:51
6,1,101,5.0,964980868,2000-07-30 18:14:28
7,1,110,4.0,964982176,2000-07-30 18:36:16
8,1,151,5.0,964984041,2000-07-30 19:07:21
9,1,157,5.0,964984100,2000-07-30 19:08:20
10,1,163,5.0,964983650,2000-07-30 19:00:50


In [ ]:
# STEP 2.2 - Select users with enough relevant movies

relevant_per_user = (
    relevant_ratings
    .groupby("userId")
    .size()
    .reset_index(name="relevant_movie_count")
)

eligible_users = relevant_per_user[
    relevant_per_user["relevant_movie_count"] >= 10
].copy()

print("Total users:", ratings["userId"].nunique())
print("Users with relevant ratings:", relevant_ratings["userId"].nunique())
print("Eligible users for evaluation:", len(eligible_users))

display(
    eligible_users
    .sort_values("relevant_movie_count", ascending=False)
    .head(15)
)

Total users: 610
Users with relevant ratings: 609
Eligible users for evaluation: 579


,userId,relevant_movie_count
413,414,1227
472,474,787
379,380,669
608,610,614
604,606,613
601,603,563
104,105,554
248,249,511
181,182,488
446,448,455


In [ ]:
# STEP 2.3 - Leave-one-out train/test split

eligible_user_ids = set(eligible_users["userId"])

evaluation_ratings = ratings[
    ratings["userId"].isin(eligible_user_ids)
].copy()

evaluation_ratings = evaluation_ratings.sort_values(
    ["userId", "timestamp"]
)

test_rows = (
    evaluation_ratings[
        evaluation_ratings["rating"] >= RELEVANCE_THRESHOLD
    ]
    .groupby("userId")
    .tail(1)
)

test_indices = test_rows.index

train_ratings = evaluation_ratings.drop(index=test_indices).copy()
test_ratings = evaluation_ratings.loc[test_indices].copy()

print("Training ratings:", len(train_ratings))
print("Test ratings:", len(test_ratings))

print("\nUsers in training set:",
      train_ratings["userId"].nunique())

print("Users in test set:",
      test_ratings["userId"].nunique())

display(test_ratings.head(10))

Training ratings: 99492
Test ratings: 579

Users in training set: 579
Users in test set: 579


,userId,movieId,rating,timestamp,datetime
161,1,2492,4.0,965719662,2000-08-08 07:27:42
247,2,80489,4.5,1445715340,2015-10-24 19:35:40
283,3,3024,4.5,1306464054,2011-05-27 02:40:54
502,4,4246,4.0,1007574542,2001-12-05 17:49:02
545,5,474,4.0,847435337,1996-11-08 06:42:17
839,6,780,5.0,845556915,1996-10-17 12:55:15
1022,7,49272,4.5,1165876367,2006-12-11 22:32:47
1040,8,186,4.0,839463856,1996-08-08 00:24:16
1087,9,2012,4.0,1044657237,2003-02-07 22:33:57
1153,10,7149,4.0,1455399533,2016-02-13 21:38:53


In [ ]:
# STEP 2.4 - Validate train/test split

print("Duplicate test users:",
      test_ratings["userId"].duplicated().sum())

print("Minimum test rating:",
      test_ratings["rating"].min())

print("Maximum test rating:",
      test_ratings["rating"].max())

print(
    "Every test movie is relevant:",
    (test_ratings["rating"] >= RELEVANCE_THRESHOLD).all()
)

print(
    "One test movie per user:",
    test_ratings["userId"].nunique() == len(test_ratings)
)

Duplicate test users: 0
Minimum test rating: 4.0
Maximum test rating: 5.0
Every test movie is relevant: True
One test movie per user: True


In [ ]:
# STEP 3.1 - Create hidden test movie lookup

test_movie_lookup = dict(
    zip(
        test_ratings["userId"],
        test_ratings["movieId"]
    )
)

print("Number of test users:", len(test_movie_lookup))

# Show first 10
for user_id, movie_id in list(test_movie_lookup.items())[:10]:
    movie_info = movies[
        movies["movieId"] == movie_id
    ]

    title = (
        movie_info["clean_title"].iloc[0]
        if len(movie_info) > 0
        else "Unknown"
    )

    print(
        f"User {user_id} -> "
        f"Hidden Movie ID: {movie_id} -> {title}"
    )

Number of test users: 579
User 1 -> Hidden Movie ID: 2492 -> 20 Dates
User 2 -> Hidden Movie ID: 80489 -> Town, The
User 3 -> Hidden Movie ID: 3024 -> Piranha
User 4 -> Hidden Movie ID: 4246 -> Bridget Jones's Diary
User 5 -> Hidden Movie ID: 474 -> In the Line of Fire
User 6 -> Hidden Movie ID: 780 -> Independence Day (a.k.a. ID4)
User 7 -> Hidden Movie ID: 49272 -> Casino Royale
User 8 -> Hidden Movie ID: 186 -> Nine Months
User 9 -> Hidden Movie ID: 2012 -> Back to the Future Part III
User 10 -> Hidden Movie ID: 7149 -> Something's Gotta Give


In [ ]:
# STEP 3.2 - Build training history for each test user

train_user_history = (
    train_ratings
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

print("Users with training history:", len(train_user_history))

# Inspect first 10 test users
for user_id in list(test_movie_lookup.keys())[:10]:

    watched = train_user_history.get(user_id, set())
    hidden_movie = test_movie_lookup[user_id]

    print(
        f"User {user_id}: "
        f"{len(watched)} training movies | "
        f"Hidden movie {hidden_movie} | "
        f"Hidden movie in training? {hidden_movie in watched}"
    )

Users with training history: 579
User 1: 231 training movies | Hidden movie 2492 | Hidden movie in training? False
User 2: 28 training movies | Hidden movie 80489 | Hidden movie in training? False
User 3: 38 training movies | Hidden movie 3024 | Hidden movie in training? False
User 4: 215 training movies | Hidden movie 4246 | Hidden movie in training? False
User 5: 43 training movies | Hidden movie 474 | Hidden movie in training? False
User 6: 313 training movies | Hidden movie 780 | Hidden movie in training? False
User 7: 151 training movies | Hidden movie 49272 | Hidden movie in training? False
User 8: 46 training movies | Hidden movie 186 | Hidden movie in training? False
User 9: 45 training movies | Hidden movie 2012 | Hidden movie in training? False
User 10: 139 training movies | Hidden movie 7149 | Hidden movie in training? False


In [ ]:
# STEP 3.3 - Generate Top-K recommendations for evaluation

def get_top_k_recommendations(
    user_id,
    k=10,
    min_rating=4.0
):
    """
    Generate Top-K candidate recommendations for offline evaluation.
    Excludes movies already present in the user's training history.
    """

    # Get movies already watched in TRAINING data
    watched_movies = train_user_history.get(user_id, set())

    # Start with movies meeting minimum quality requirement
    candidates = movies[
        movies["vote_average"].fillna(0) >= min_rating
    ].copy()

    # Remove movies already watched
    candidates = candidates[
        ~candidates["movieId"].isin(watched_movies)
    ].copy()

    # Make numeric columns safe
    candidates["vote_average"] = (
        candidates["vote_average"]
        .fillna(0)
    )

    candidates["tmdb_popularity"] = (
        candidates["tmdb_popularity"]
        .fillna(0)
    )

    # Rank candidates using rating + popularity
    candidates = candidates.sort_values(
        ["vote_average", "tmdb_popularity"],
        ascending=[False, False]
    )

    return candidates.head(k)

In [ ]:
# STEP 3.4 - Test Top-K recommendation generation

sample_user = list(test_movie_lookup.keys())[0]

sample_recommendations = get_top_k_recommendations(
    sample_user,
    k=10
)

hidden_movie = test_movie_lookup[sample_user]

print("Test User:", sample_user)
print("Hidden Movie ID:", hidden_movie)

hidden_info = movies[
    movies["movieId"] == hidden_movie
]

if len(hidden_info) > 0:
    print(
        "Hidden Movie:",
        hidden_info["clean_title"].iloc[0]
    )

print("\nTop-10 Recommendations:")

display(
    sample_recommendations[
        [
            "movieId",
            "clean_title",
            "genres",
            "vote_average",
            "tmdb_popularity"
        ]
    ]
)

recommended_ids = set(
    sample_recommendations["movieId"]
)

print(
    "\nHidden movie found in Top-10:",
    hidden_movie in recommended_ids
)

Test User: 1
Hidden Movie ID: 2492
Hidden Movie: 20 Dates

Top-10 Recommendations:


,movieId,clean_title,genres,vote_average,tmdb_popularity
9541,172591,The Godfather Trilogy: 1972-1990,(no genres listed),8.896,0.8660
277,318,"Shawshank Redemption, The",Crime|Drama,8.726,59.5022
659,858,"Godfather, The",Crime|Drama,8.687,28.5584
922,1221,"Godfather: Part II, The",Crime|Drama,8.572,19.3643
905,1203,12 Angry Men,Drama,8.564,16.2413
3984,5618,Spirited Away (Sen to Chihiro no kamikakushi),Adventure|Animation|Fantasy,8.535,25.3855
6710,58559,"Dark Knight, The",Action|Crime|Drama|IMAX,8.533,33.8159
4800,7153,"Lord of the Rings: The Return of the King, The",Action|Adventure|Drama|Fantasy,8.501,30.0568
8376,109487,Interstellar,Sci-Fi|IMAX,8.482,59.3212
9381,163134,Your Name.,Animation|Drama|Fantasy|Romance,8.481,21.2840



Hidden movie found in Top-10: False


In [ ]:
# STEP 3.5 - Calculate baseline Hit Rate@10

K = 10

hits = 0
evaluation_results = []

for user_id, hidden_movie in test_movie_lookup.items():

    recommendations = get_top_k_recommendations(
        user_id,
        k=K
    )

    recommended_ids = recommendations["movieId"].tolist()

    hit = int(hidden_movie in recommended_ids)

    hits += hit

    evaluation_results.append({
        "userId": user_id,
        "hidden_movieId": hidden_movie,
        "hit": hit
    })


baseline_hit_rate = hits / len(test_movie_lookup)

print("Users evaluated:", len(test_movie_lookup))
print("Successful hits:", hits)

print(
    f"Baseline Hit Rate@{K}: "
    f"{baseline_hit_rate:.4f}"
)

print(
    f"Baseline Hit Rate@{K}: "
    f"{baseline_hit_rate * 100:.2f}%"
)

Users evaluated: 579
Successful hits: 22
Baseline Hit Rate@10: 0.0380
Baseline Hit Rate@10: 3.80%


In [ ]:
# STEP 3.6 - Calculate baseline ranking metrics

import numpy as np
import pandas as pd

K = 10

precision_scores = []
recall_scores = []
ndcg_scores = []

for user_id, hidden_movie in test_movie_lookup.items():

    recommendations = get_top_k_recommendations(
        user_id,
        k=K
    )

    recommended_ids = recommendations["movieId"].tolist()

    # ---------- Precision@K ----------
    if hidden_movie in recommended_ids:
        precision = 1 / K
    else:
        precision = 0

    # ---------- Recall@K ----------
    recall = int(hidden_movie in recommended_ids)

    # ---------- NDCG@K ----------
    if hidden_movie in recommended_ids:

        rank = recommended_ids.index(hidden_movie) + 1

        ndcg = 1 / np.log2(rank + 1)

    else:
        ndcg = 0

    precision_scores.append(precision)
    recall_scores.append(recall)
    ndcg_scores.append(ndcg)


baseline_precision = np.mean(precision_scores)
baseline_recall = np.mean(recall_scores)
baseline_ndcg = np.mean(ndcg_scores)

print("CineSense AI - Baseline Evaluation")
print("-" * 40)

print(f"Precision@{K}: {baseline_precision:.4f}")
print(f"Recall@{K}:    {baseline_recall:.4f}")
print(f"Hit Rate@{K}:  {baseline_hit_rate:.4f}")
print(f"NDCG@{K}:      {baseline_ndcg:.4f}")

print("\nPercentage Form")
print("-" * 40)

print(f"Precision@{K}: {baseline_precision * 100:.2f}%")
print(f"Recall@{K}:    {baseline_recall * 100:.2f}%")
print(f"Hit Rate@{K}:  {baseline_hit_rate * 100:.2f}%")
print(f"NDCG@{K}:      {baseline_ndcg * 100:.2f}%")

CineSense AI - Baseline Evaluation
----------------------------------------
Precision@10: 0.0038
Recall@10:    0.0380
Hit Rate@10:  0.0380
NDCG@10:      0.0178

Percentage Form
----------------------------------------
Precision@10: 0.38%
Recall@10:    3.80%
Hit Rate@10:  3.80%
NDCG@10:      1.78%


In [ ]:
# STEP 4.1 - Build user genre preference profiles

train_history_with_genres = train_ratings.merge(
    movie_genres,
    on="movieId",
    how="left"
)

# Keep ratings where genre exists
train_history_with_genres = train_history_with_genres.dropna(
    subset=["genre"]
)

# Average rating each user gives to each genre
user_genre_preferences = (
    train_history_with_genres
    .groupby(["userId", "genre"])
    .agg(
        average_rating=("rating", "mean"),
        movies_rated=("movieId", "nunique")
    )
    .reset_index()
)

# Overall mean rating of each user
user_mean_ratings = (
    train_ratings
    .groupby("userId")["rating"]
    .mean()
    .to_dict()
)

# Preference score:
# positive = user likes genre more than their average
user_genre_preferences["preference_score"] = (
    user_genre_preferences.apply(
        lambda row:
        row["average_rating"]
        - user_mean_ratings.get(row["userId"], 0),
        axis=1
    )
)

print(
    "User-genre preference records:",
    len(user_genre_preferences)
)

display(
    user_genre_preferences
    .sort_values(
        ["userId", "preference_score"],
        ascending=[True, False]
    )
    .head(20)
)

User-genre preference records: 9552


,userId,genre,average_rating,movies_rated,preference_score
8,1,Film-Noir,5.000000,1,0.632035
2,1,Animation,4.689655,29,0.321690
10,1,Musical,4.681818,22,0.313853
3,1,Children,4.547619,42,0.179654
6,1,Drama,4.529412,68,0.161446
15,1,War,4.500000,22,0.132035
1,1,Adventure,4.388235,85,0.020270
5,1,Crime,4.355556,45,-0.012410
0,1,Action,4.322222,90,-0.045743
12,1,Romance,4.320000,25,-0.047965


In [ ]:
# STEP 4.2 - Personalized Top-K recommendation function

def get_personalized_top_k(user_id, k=10):

    # Movies already watched in TRAINING data
    watched_movies = train_user_history.get(user_id, set())

    # Get this user's genre preferences
    user_preferences = user_genre_preferences[
        user_genre_preferences["userId"] == user_id
    ]

    preference_dict = dict(
        zip(
            user_preferences["genre"],
            user_preferences["preference_score"]
        )
    )

    # Candidate movies
    candidates = movies[
        ~movies["movieId"].isin(watched_movies)
    ].copy()

    # Safe numeric values
    candidates["vote_average"] = (
        candidates["vote_average"].fillna(0)
    )

    candidates["tmdb_popularity"] = (
        candidates["tmdb_popularity"].fillna(0)
    )

    # Calculate personalization score
    def movie_preference_score(genres):

        if pd.isna(genres):
            return 0

        genre_list = str(genres).split("|")

        scores = [
            preference_dict.get(g, 0)
            for g in genre_list
        ]

        if len(scores) == 0:
            return 0

        return max(scores)

    candidates["personalization_score"] = (
        candidates["genres"]
        .apply(movie_preference_score)
    )

    # Normalize rating
    candidates["rating_normalized"] = (
        candidates["vote_average"] / 10
    )

    # Combined personalized score
    candidates["personalized_score"] = (
        0.70 * candidates["rating_normalized"]
        +
        0.30 * candidates["personalization_score"]
    )

    candidates = candidates.sort_values(
        "personalized_score",
        ascending=False
    )

    return candidates.head(k)

In [ ]:
# STEP 4.3 - Test personalized recommendations

sample_user = 1

personalized_top10 = get_personalized_top_k(
    sample_user,
    k=10
)

hidden_movie = test_movie_lookup[sample_user]

print("User:", sample_user)
print("Hidden Movie ID:", hidden_movie)

display(
    personalized_top10[
        [
            "movieId",
            "clean_title",
            "genres",
            "vote_average",
            "personalization_score",
            "personalized_score"
        ]
    ]
)

print(
    "\nHidden movie found:",
    hidden_movie in personalized_top10["movieId"].values
)

User: 1
Hidden Movie ID: 2492


,movieId,clean_title,genres,vote_average,personalization_score,personalized_score
5058,7926,High and Low (Tengoku to jigoku),Crime|Drama|Film-Noir|Thriller,8.363,0.632035,0.77502
704,922,Sunset Blvd. (a.k.a. Sunset Boulevard),Drama|Film-Noir|Romance,8.275,0.632035,0.76886
5786,31545,"Trou, Le (Hole, The) (Night Watch, The)",Crime|Film-Noir,8.233,0.632035,0.76592
959,1260,M,Crime|Film-Noir|Thriller,8.100,0.632035,0.75661
2568,3435,Double Indemnity,Crime|Drama|Film-Noir,8.084,0.632035,0.75549
951,1252,Chinatown,Crime|Film-Noir|Mystery|Thriller,7.905,0.632035,0.74296
913,1212,"Third Man, The",Film-Noir|Mystery|Thriller,7.900,0.632035,0.74261
4698,7013,"Night of the Hunter, The",Drama|Film-Noir|Thriller,7.859,0.632035,0.73974
3544,4848,Mulholland Drive,Crime|Drama|Film-Noir|Mystery|Thriller,7.800,0.632035,0.73561
5096,8044,I Am a Fugitive from a Chain Gang,Crime|Drama|Film-Noir,7.780,0.632035,0.73421



Hidden movie found: False


In [ ]:
# STEP 4.4 - Evaluate personalized recommender

K = 10

personalized_hits = 0
personalized_precision_scores = []
personalized_recall_scores = []
personalized_ndcg_scores = []

for user_id, hidden_movie in test_movie_lookup.items():

    recommendations = get_personalized_top_k(
        user_id,
        k=K
    )

    recommended_ids = recommendations["movieId"].tolist()

    # Check whether hidden movie was recovered
    if hidden_movie in recommended_ids:

        personalized_hits += 1

        # One relevant item among K recommendations
        precision = 1 / K
        recall = 1

        # Rank of hidden movie
        rank = recommended_ids.index(hidden_movie) + 1

        ndcg = 1 / np.log2(rank + 1)

    else:

        precision = 0
        recall = 0
        ndcg = 0

    personalized_precision_scores.append(precision)
    personalized_recall_scores.append(recall)
    personalized_ndcg_scores.append(ndcg)


personalized_precision = np.mean(
    personalized_precision_scores
)

personalized_recall = np.mean(
    personalized_recall_scores
)

personalized_hit_rate = (
    personalized_hits / len(test_movie_lookup)
)

personalized_ndcg = np.mean(
    personalized_ndcg_scores
)


print("CineSense AI - Personalized Evaluation")
print("-" * 45)

print("Users evaluated:", len(test_movie_lookup))
print("Successful hits:", personalized_hits)

print(f"\nPrecision@{K}: {personalized_precision:.4f}")
print(f"Recall@{K}:    {personalized_recall:.4f}")
print(f"Hit Rate@{K}:  {personalized_hit_rate:.4f}")
print(f"NDCG@{K}:      {personalized_ndcg:.4f}")

print("\nPercentage Form")
print("-" * 45)

print(f"Precision@{K}: {personalized_precision * 100:.2f}%")
print(f"Recall@{K}:    {personalized_recall * 100:.2f}%")
print(f"Hit Rate@{K}:  {personalized_hit_rate * 100:.2f}%")
print(f"NDCG@{K}:      {personalized_ndcg * 100:.2f}%")

CineSense AI - Personalized Evaluation
---------------------------------------------
Users evaluated: 579
Successful hits: 10

Precision@10: 0.0017
Recall@10:    0.0173
Hit Rate@10:  0.0173
NDCG@10:      0.0102

Percentage Form
---------------------------------------------
Precision@10: 0.17%
Recall@10:    1.73%
Hit Rate@10:  1.73%
NDCG@10:      1.02%


In [ ]:
# STEP 4.5 - Confidence-weighted genre preferences

user_genre_preferences["confidence"] = (
    user_genre_preferences["movies_rated"] /
    (user_genre_preferences["movies_rated"] + 5)
)

user_genre_preferences["weighted_preference"] = (
    user_genre_preferences["preference_score"]
    * user_genre_preferences["confidence"]
)

display(
    user_genre_preferences[
        user_genre_preferences["userId"] == 1
    ][
        [
            "genre",
            "average_rating",
            "movies_rated",
            "preference_score",
            "confidence",
            "weighted_preference"
        ]
    ]
    .sort_values(
        "weighted_preference",
        ascending=False
    )
)

,genre,average_rating,movies_rated,preference_score,confidence,weighted_preference
2,Animation,4.689655,29,0.321690,0.852941,0.274382
10,Musical,4.681818,22,0.313853,0.814815,0.255732
3,Children,4.547619,42,0.179654,0.893617,0.160542
6,Drama,4.529412,68,0.161446,0.931507,0.150388
15,War,4.500000,22,0.132035,0.814815,0.107584
8,Film-Noir,5.000000,1,0.632035,0.166667,0.105339
1,Adventure,4.388235,85,0.020270,0.944444,0.019144
5,Crime,4.355556,45,-0.012410,0.900000,-0.011169
12,Romance,4.320000,25,-0.047965,0.833333,-0.039971
0,Action,4.322222,90,-0.045743,0.947368,-0.043336


In [ ]:
# STEP 4.6 - Improved personalized recommender

def get_improved_personalized_top_k(user_id, k=10):

    watched_movies = train_user_history.get(user_id, set())

    # Get this user's weighted genre preferences
    user_preferences = user_genre_preferences[
        user_genre_preferences["userId"] == user_id
    ]

    preference_dict = dict(
        zip(
            user_preferences["genre"],
            user_preferences["weighted_preference"]
        )
    )

    # Remove movies already watched
    candidates = movies[
        ~movies["movieId"].isin(watched_movies)
    ].copy()

    candidates["vote_average"] = (
        candidates["vote_average"].fillna(0)
    )

    candidates["tmdb_popularity"] = (
        candidates["tmdb_popularity"].fillna(0)
    )

    # Calculate average weighted genre preference
    def calculate_genre_score(genres):

        if pd.isna(genres):
            return 0.0

        genres_list = str(genres).split("|")

        scores = [
            preference_dict.get(genre, 0.0)
            for genre in genres_list
        ]

        if not scores:
            return 0.0

        return np.mean(scores)

    candidates["genre_preference_score"] = (
        candidates["genres"].apply(
            calculate_genre_score
        )
    )

    # Normalize movie rating
    candidates["rating_normalized"] = (
        candidates["vote_average"] / 10
    )

    # Improved hybrid score
    candidates["improved_score"] = (
        0.80 * candidates["rating_normalized"]
        +
        0.20 * candidates["genre_preference_score"]
    )

    candidates = candidates.sort_values(
        ["improved_score", "tmdb_popularity"],
        ascending=[False, False]
    )

    return candidates.head(k)

In [ ]:
# STEP 4.7 - Test improved personalization

sample_user = 1

improved_top10 = get_improved_personalized_top_k(
    sample_user,
    k=10
)

hidden_movie = test_movie_lookup[sample_user]

print("User:", sample_user)
print("Hidden Movie ID:", hidden_movie)

display(
    improved_top10[
        [
            "movieId",
            "clean_title",
            "genres",
            "vote_average",
            "genre_preference_score",
            "improved_score"
        ]
    ]
)

print(
    "\nHidden movie found:",
    hidden_movie in improved_top10["movieId"].values
)

User: 1
Hidden Movie ID: 2492


,movieId,clean_title,genres,vote_average,genre_preference_score,improved_score
905,1203,12 Angry Men,Drama,8.564,0.150388,0.715198
277,318,"Shawshank Redemption, The",Crime|Drama,8.726,0.069610,0.712002
9541,172591,The Godfather Trilogy: 1972-1990,(no genres listed),8.896,0.000000,0.711680
4025,5690,Grave of the Fireflies (Hotaru no haka),Animation|Drama|War,8.440,0.177452,0.710690
659,858,"Godfather, The",Crime|Drama,8.687,0.069610,0.708882
9340,160718,Piper,Animation,8.144,0.274382,0.706396
5450,26082,Harakiri (Seppuku),Drama,8.441,0.150388,0.705358
878,1172,Cinema Paradiso (Nuovo cinema Paradiso),Drama,8.429,0.150388,0.704398
896,1193,One Flew Over the Cuckoo's Nest,Drama,8.409,0.150388,0.702798
8466,112552,Whiplash,Drama,8.376,0.150388,0.700158



Hidden movie found: False


In [ ]:
def get_improved_personalized_top_k(user_id, k=10):

    watched_movies = train_user_history.get(user_id, set())

    user_preferences = user_genre_preferences[
        user_genre_preferences["userId"] == user_id
    ]

    preference_dict = dict(
        zip(
            user_preferences["genre"],
            user_preferences["weighted_preference"]
        )
    )

    candidates = movies[
        ~movies["movieId"].isin(watched_movies)
    ].copy()

    candidates["vote_average"] = candidates["vote_average"].fillna(0)
    candidates["tmdb_popularity"] = candidates["tmdb_popularity"].fillna(0)

    def calculate_genre_score(genres):

        if pd.isna(genres):
            return 0.0

        genres_list = str(genres).split("|")

        scores = [
            preference_dict.get(g, 0.0)
            for g in genres_list
        ]

        return np.mean(scores) if scores else 0.0

    candidates["genre_preference_score"] = (
        candidates["genres"].apply(calculate_genre_score)
    )

    candidates["rating_normalized"] = (
        candidates["vote_average"] / 10
    )

    candidates["improved_score"] = (
        0.80 * candidates["rating_normalized"]
        + 0.20 * candidates["genre_preference_score"]
    )

    candidates = candidates.sort_values(
        ["improved_score", "tmdb_popularity"],
        ascending=[False, False]
    )

    return candidates.head(k)

In [ ]:
K = 10

improved_hits = 0
improved_precision_scores = []
improved_recall_scores = []
improved_ndcg_scores = []

for user_id, hidden_movie in test_movie_lookup.items():

    recommendations = get_improved_personalized_top_k(
        user_id,
        k=K
    )

    recommended_ids = recommendations["movieId"].tolist()

    if hidden_movie in recommended_ids:

        improved_hits += 1

        precision = 1 / K
        recall = 1

        rank = recommended_ids.index(hidden_movie) + 1
        ndcg = 1 / np.log2(rank + 1)

    else:

        precision = 0
        recall = 0
        ndcg = 0

    improved_precision_scores.append(precision)
    improved_recall_scores.append(recall)
    improved_ndcg_scores.append(ndcg)


improved_precision = np.mean(improved_precision_scores)
improved_recall = np.mean(improved_recall_scores)

improved_hit_rate = (
    improved_hits / len(test_movie_lookup)
)

improved_ndcg = np.mean(improved_ndcg_scores)


print("CineSense AI - Improved Personalized Evaluation")
print("-" * 50)

print("Users evaluated:", len(test_movie_lookup))
print("Successful hits:", improved_hits)

print(f"Precision@{K}: {improved_precision:.4f}")
print(f"Recall@{K}:    {improved_recall:.4f}")
print(f"Hit Rate@{K}:  {improved_hit_rate:.4f}")
print(f"NDCG@{K}:      {improved_ndcg:.4f}")

CineSense AI - Improved Personalized Evaluation
--------------------------------------------------
Users evaluated: 579
Successful hits: 15
Precision@10: 0.0026
Recall@10:    0.0259
Hit Rate@10:  0.0259
NDCG@10:      0.0112


In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Baseline",
        "Basic Personalization",
        "Improved Personalization"
    ],

    "Precision@10": [
        baseline_precision,
        personalized_precision,
        improved_precision
    ],

    "Recall@10": [
        baseline_recall,
        personalized_recall,
        improved_recall
    ],

    "Hit Rate@10": [
        baseline_hit_rate,
        personalized_hit_rate,
        improved_hit_rate
    ],

    "NDCG@10": [
        baseline_ndcg,
        personalized_ndcg,
        improved_ndcg
    ]
})

display(comparison)

,Model,Precision@10,Recall@10,Hit Rate@10,NDCG@10
0,Baseline,0.003800,0.037997,0.037997,0.017774
1,Basic Personalization,0.001727,0.017271,0.017271,0.010157
2,Improved Personalization,0.002591,0.025907,0.025907,0.011247


In [ ]:
comparison_percent = comparison.copy()

metric_columns = [
    "Precision@10",
    "Recall@10",
    "Hit Rate@10",
    "NDCG@10"
]

comparison_percent[metric_columns] = (
    comparison_percent[metric_columns] * 100
)

display(
    comparison_percent.style.format({
        "Precision@10": "{:.2f}%",
        "Recall@10": "{:.2f}%",
        "Hit Rate@10": "{:.2f}%",
        "NDCG@10": "{:.2f}%"
    })
)

,Model,Precision@10,Recall@10,Hit Rate@10,NDCG@10
0,Baseline,0.38%,3.80%,3.80%,1.78%
1,Basic Personalization,0.17%,1.73%,1.73%,1.02%
2,Improved Personalization,0.26%,2.59%,2.59%,1.12%


In [ ]:
def evaluate_recommender(recommend_function, k):

    hits = 0
    precision_list = []
    recall_list = []
    ndcg_list = []

    for user_id, hidden_movie in test_movie_lookup.items():

        recommendations = recommend_function(
            user_id,
            k=k
        )

        recommended_ids = recommendations["movieId"].tolist()

        if hidden_movie in recommended_ids:

            hits += 1

            precision = 1 / k
            recall = 1

            rank = recommended_ids.index(hidden_movie) + 1
            ndcg = 1 / np.log2(rank + 1)

        else:

            precision = 0
            recall = 0
            ndcg = 0

        precision_list.append(precision)
        recall_list.append(recall)
        ndcg_list.append(ndcg)

    return {
        "K": k,
        "Precision": np.mean(precision_list),
        "Recall": np.mean(recall_list),
        "Hit_Rate": hits / len(test_movie_lookup),
        "NDCG": np.mean(ndcg_list)
    }

In [ ]:
k_results = []

for k in [5, 10, 20]:

    result = evaluate_recommender(
        get_improved_personalized_top_k,
        k
    )

    k_results.append(result)

k_results_df = pd.DataFrame(k_results)

display(k_results_df)

,K,Precision,Recall,Hit_Rate,NDCG
0,5,0.002763,0.013817,0.013817,0.007272
1,10,0.002591,0.025907,0.025907,0.011247
2,20,0.002159,0.043178,0.043178,0.015688


In [ ]:
all_recommended_movies = set()

for user_id in test_movie_lookup.keys():

    recs = get_improved_personalized_top_k(
        user_id,
        k=10
    )

    all_recommended_movies.update(
        recs["movieId"].tolist()
    )


catalog_size = movies["movieId"].nunique()

coverage = (
    len(all_recommended_movies)
    / catalog_size
)

print("Unique recommended movies:",
      len(all_recommended_movies))

print("Total catalog movies:",
      catalog_size)

print(
    f"Catalog Coverage@10: "
    f"{coverage * 100:.2f}%"
)

Unique recommended movies: 207
Total catalog movies: 9742
Catalog Coverage@10: 2.12%


In [ ]:
def genre_diversity(recommendations):

    genre_set = set()

    for genres in recommendations["genres"].dropna():

        genre_set.update(
            str(genres).split("|")
        )

    return len(genre_set)


diversity_scores = []

for user_id in test_movie_lookup.keys():

    recs = get_improved_personalized_top_k(
        user_id,
        k=10
    )

    diversity_scores.append(
        genre_diversity(recs)
    )


average_genre_diversity = np.mean(
    diversity_scores
)

print(
    "Average unique genres in Top-10:",
    round(average_genre_diversity, 2)
)

Average unique genres in Top-10: 6.34


In [ ]:
comparison.to_csv(
    "model_comparison.csv",
    index=False
)

k_results_df.to_csv(
    "topk_evaluation.csv",
    index=False
)

print("Saved:")
print("model_comparison.csv")
print("topk_evaluation.csv")

Saved:
model_comparison.csv
topk_evaluation.csv
